# PAD-UFES-20 Colab Training Launcher

Run this notebook in Google Colab with a GPU runtime. It prepares the PAD-UFES-20 dataset, creates patient-safe splits, checks DagsHub MLflow connectivity, then launches the image-only baseline, optional hyperparameter sweep, or image-plus-metadata multimodal baseline CLIs.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import os
import subprocess

try:
    from google.colab import userdata
except ImportError:
    userdata = None


def get_config(name, default=None):
    value = os.environ.get(name)
    if value:
        return value
    if userdata is not None:
        try:
            return userdata.get(name) or default
        except Exception:
            return default
    return default


def export_config(name, default=None, required=False):
    value = get_config(name, default)
    if required and not value:
        raise RuntimeError(f'Set {name} in Colab Secrets before training.')
    if value:
        os.environ[name] = str(value)
    return value


REPO_URL = 'https://github.com/SalmaneSossey/mlops-teledermatology.git'
BRANCH = 'main'
REPO_DIR = Path('/content/mlops-teledermatology')
HF_DATASET_REPO = get_config('PAD_UFES20_HF_REPO_ID', 'SalmaneExploring/pad-ufes-20')
DATA_ROOT = Path('/content/pad_ufes_20')
IMAGES_DIR = DATA_ROOT / 'all_images'
IMAGE_BASELINE_OUTPUT_DIR = Path('/content/drive/MyDrive/mlops-teledermatology/runs/image_baseline')
MULTIMODAL_OUTPUT_DIR = Path('/content/drive/MyDrive/mlops-teledermatology/runs/multimodal_baseline')
RUN_IMAGE_BASELINE = False
RUN_HPARAM_SWEEP = False
SWEEP_MAX_TRIALS = 8
RETRAIN_BEST_AFTER_SWEEP = False
RUN_MULTIMODAL_BASELINE = True
MULTIMODAL_EPOCHS = 8
MULTIMODAL_BATCH_SIZE = 32
MULTIMODAL_LOSS_TYPE = 'weighted_cross_entropy'
MULTIMODAL_SAMPLER = 'shuffle'
MULTIMODAL_AUGMENT_STRENGTH = 'current'

export_config('DAGSHUB_TOKEN', required=True)
export_config('DAGSHUB_USERNAME')
export_config('DAGSHUB_REPO_OWNER', 'SalmaneSossey')
export_config('DAGSHUB_REPO_NAME', 'mlops-teledermatology')
export_config('DAGSHUB_MLFLOW_TRACKING_URI')

IMAGE_BASELINE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MULTIMODAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if REPO_DIR.exists():
    subprocess.run(['git', 'remote', 'set-url', 'origin', REPO_URL], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'checkout', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=REPO_DIR, check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO_DIR)], cwd='/content', check=True)

os.chdir(REPO_DIR)
print('Working directory:', Path.cwd())
print('Hugging Face dataset:', HF_DATASET_REPO)
print('PAD-UFES-20 data root:', DATA_ROOT)
print('PAD-UFES-20 images dir:', IMAGES_DIR)
print('Image baseline output dir:', IMAGE_BASELINE_OUTPUT_DIR)
print('Multimodal output dir:', MULTIMODAL_OUTPUT_DIR)
print('MLflow tracking URI:', os.environ.get('DAGSHUB_MLFLOW_TRACKING_URI') or f"https://dagshub.com/{os.environ['DAGSHUB_REPO_OWNER']}/{os.environ['DAGSHUB_REPO_NAME']}.mlflow")


In [ ]:
!pip -q install mlflow huggingface_hub scikit-learn


## DagsHub MLflow Check


In [ ]:
import mlflow
from mlflow.tracking import MlflowClient

tracking_uri = os.environ.get('DAGSHUB_MLFLOW_TRACKING_URI') or f"https://dagshub.com/{os.environ['DAGSHUB_REPO_OWNER']}/{os.environ['DAGSHUB_REPO_NAME']}.mlflow"
mlflow.set_tracking_uri(tracking_uri)
experiments = MlflowClient(tracking_uri).search_experiments()
print('DagsHub token available:', bool(os.environ.get('DAGSHUB_TOKEN')))
print('MLflow tracking URI:', tracking_uri)
print('Existing experiments:', [(experiment.experiment_id, experiment.name) for experiment in experiments])


## Runtime Check


In [ ]:
import torch

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
elif RUN_IMAGE_BASELINE or RUN_HPARAM_SWEEP or RUN_MULTIMODAL_BASELINE:
    raise RuntimeError('Select a Colab GPU runtime before starting training.')


In [ ]:
!python -m src.data.download_pad_ufes_20 \
  --repo-id "{HF_DATASET_REPO}" \
  --output-dir "{DATA_ROOT}" \
  --force

!python -m src.data.make_image_splits \
  --metadata-path "{DATA_ROOT / 'metadata.csv'}" \
  --images-dir "{IMAGES_DIR}" \
  --output-dir data/processed/splits


In [ ]:
if RUN_IMAGE_BASELINE:
    subprocess.run([
        'python', '-m', 'src.training.train_image_baseline',
        '--images-dir', str(IMAGES_DIR),
        '--splits-dir', 'data/processed/splits',
        '--output-dir', str(IMAGE_BASELINE_OUTPUT_DIR),
        '--hf-dataset-repo', HF_DATASET_REPO,
    ], check=True)
else:
    print('Skipping image baseline. Set RUN_IMAGE_BASELINE = True to rerun it.')


## Optional Hyperparameter Sweep

Set `RUN_HPARAM_SWEEP = True` in the setup cell after confirming DagsHub credentials and GPU runtime availability.


In [ ]:
if RUN_HPARAM_SWEEP:
    command = [
        'python', '-m', 'src.training.run_hparam_sweep',
        '--images-dir', str(IMAGES_DIR),
        '--splits-dir', 'data/processed/splits',
        '--output-dir', str(IMAGE_BASELINE_OUTPUT_DIR),
        '--max-trials', str(SWEEP_MAX_TRIALS),
        '--hf-dataset-repo', HF_DATASET_REPO,
    ]
    if RETRAIN_BEST_AFTER_SWEEP:
        command.append('--retrain-best')
    subprocess.run(command, check=True)
else:
    subprocess.run([
        'python', '-m', 'src.training.run_hparam_sweep',
        '--images-dir', str(IMAGES_DIR),
        '--splits-dir', 'data/processed/splits',
        '--output-dir', str(IMAGE_BASELINE_OUTPUT_DIR),
        '--max-trials', '3',
        '--dry-run',
    ], check=True)


## Optional Image Plus Metadata Baseline

Set `RUN_MULTIMODAL_BASELINE = True` in the setup cell after reviewing the image-only and metadata-only baselines.


In [ ]:
if RUN_MULTIMODAL_BASELINE:
    subprocess.run([
        'python', '-m', 'src.training.train_multimodal_baseline',
        '--images-dir', str(IMAGES_DIR),
        '--metadata-path', str(DATA_ROOT / 'metadata.csv'),
        '--splits-dir', 'data/processed/splits',
        '--output-dir', str(MULTIMODAL_OUTPUT_DIR),
        '--hf-dataset-repo', HF_DATASET_REPO,
        '--epochs', str(MULTIMODAL_EPOCHS),
        '--batch-size', str(MULTIMODAL_BATCH_SIZE),
        '--loss-type', MULTIMODAL_LOSS_TYPE,
        '--sampler', MULTIMODAL_SAMPLER,
        '--augment-strength', MULTIMODAL_AUGMENT_STRENGTH,
    ], check=True)
else:
    subprocess.run([
        'python', '-m', 'src.training.train_multimodal_baseline',
        '--help',
    ], check=True)
